# SQL Ratio Interview Problems

Companion to the [complete article](https://todatabeyond.com/blog/how-to-ace-the-sql-ratio-problems-in-data-science-interviews). Run all cells in order in a Python 3.11+ Jupyter/Colab kernel with SQLite 3.25+. Standard-library sqlite3 executes real SQL in isolated in-memory databases. The small %%sql helper is registered explicitly below; no extra SQL extension, server, credentials, files or data downloads.

Input rows are transcribed exactly from the article's two original table images. Convert original day/month/year dates to ISO before ordering them. The subscription data has three users, two premium; delivery has seven orders from three customers, with only customer 2's earliest order immediate. These original source examples are not substituted by invented datasets. Additional synthetic fixtures live only in regression tests.

Data contract: one nonnull unique user_id per subscription; Premium text is exactly True/False; nonnull ISO order/preferred dates; at most one order per customer per date; preferred date is not before order date. Duplicate same-day orders are rejected, not arbitrarily tie-broken or silently counted twice. Real systems with tied first-order timestamps need an explicit business tie policy/order key. These examples do not claim to handle that different schema.

All seven source SQL examples are maintained. Replace reserved TABLE placeholder by subscription, add NULLIF zero-denominator guards and complete the standalone CTE with SELECT. The immediate subquery matches each minimum order date to its actual row instead of comparing independent column minima, then divides by distinct customers, not delivery rows. CASE WHEN averages only rank=1 rows. Preserve all three subscription techniques and both immediate-order techniques.

Outputs are fractions in [0,1], not multiplied-by-100 percentages despite the historical immediate_percentage alias. Empty populations return SQL NULL. Source prose still names the old placeholders, row denominator and integer-AVG rationale; differences are recorded in the website review rather than agent-added public notes. This run validates SQLite, not every database dialect. All exact current results are executed, not prewritten. The final regression cell closes connections; rerun setup before queries. Git outputs/counts stay empty/null.


In [ ]:
import sqlite3
from datetime import date, datetime, timedelta
from IPython.core.magic import register_cell_magic

assert sqlite3.sqlite_version_info >= (3, 25, 0), 'SQLite 3.25+ is required'
connection = sqlite3.connect(':memory:')
SCHEMA = """
CREATE TABLE subscription (
    user_id INTEGER NOT NULL PRIMARY KEY,
    Premium TEXT NOT NULL CHECK (Premium IN ('True', 'False'))
);
CREATE TABLE delivery (
    customer_id INTEGER NOT NULL,
    order_date TEXT NOT NULL,
    pref_delivery_date TEXT NOT NULL,
    PRIMARY KEY (customer_id, order_date),
    CHECK (date(order_date) IS NOT NULL AND date(order_date) = order_date),
    CHECK (date(pref_delivery_date) IS NOT NULL AND date(pref_delivery_date) = pref_delivery_date),
    CHECK (pref_delivery_date >= order_date)
);
"""
connection.executescript(SCHEMA)
# Exact rows transcribed from the two original article table images.
subscription_fixture = [(1, 'True'), (2, 'False'), (3, 'True')]
delivery_dates = [
    (1, '17/03/2023', '17/03/2023'),
    (2, '18/02/2023', '21/02/2023'),
    (3, '15/01/2023', '15/01/2023'),
    (1, '18/12/2022', '19/12/2022'),
    (3, '12/12/2022', '15/12/2022'),
    (1, '10/10/2022', '11/10/2022'),
    (2, '5/5/2022', '5/5/2022'),
]
def iso(day):
    return datetime.strptime(day, '%d/%m/%Y').date().isoformat()
delivery_fixture = [(customer, iso(ordered), iso(preferred))
                    for customer, ordered, preferred in delivery_dates]
connection.executemany('INSERT INTO subscription VALUES (?, ?)', subscription_fixture)
connection.executemany('INSERT INTO delivery VALUES (?, ?, ?)', delivery_fixture)
query_results = {}
query_texts = {}

def print_result(columns, rows):
    text_rows = [[str(value) if value is not None else 'NULL' for value in row] for row in rows]
    widths = [max([len(column)] + [len(row[i]) for row in text_rows]) for i, column in enumerate(columns)]
    def format_row(row):
        return ' | '.join(value.ljust(width) for value, width in zip(row, widths)).rstrip()
    print(format_row(columns))
    print('-+-'.join('-' * width for width in widths))
    for row in text_rows:
        print(format_row(row))
    print('Rows:', len(rows))

@register_cell_magic
def sql(line, cell):
    key = line.strip()
    cursor = connection.execute(cell)
    columns = [column[0] for column in cursor.description]
    rows = cursor.fetchall()
    query_texts[key] = cell.strip()
    query_results[key] = (columns, rows)
    print_result(columns, rows)


print('SQLite:', sqlite3.sqlite_version)
print('Recovered source tables: 3 users; 7 deliveries from 3 customers.')
print('Dates normalized from day/month/year to ISO; one order per customer/date.')
print('SQL magic ready. All databases are in-memory; no files, keys or downloads.')


## Query 1


In [ ]:
%%sql example-0
SELECT COUNT(user_id) * 1.0 /
       NULLIF((SELECT COUNT(user_id) FROM subscription), 0) AS ratio
FROM subscription
WHERE Premium = 'True';


## Query 2


In [ ]:
%%sql example-1
SELECT SUM(CASE WHEN Premium = 'True' THEN 1 ELSE 0 END) * 1.0 /
       NULLIF(COUNT(user_id), 0) AS ratio
FROM subscription;


## Query 3


In [ ]:
%%sql example-2
SELECT AVG(CASE WHEN Premium = 'True' THEN 1.0 ELSE 0.0 END) AS ratio
FROM subscription;


## Query 4


In [ ]:
%%sql example-3
WITH first_order_dates AS (
    SELECT customer_id, MIN(order_date) AS first_order_date
    FROM delivery
    GROUP BY customer_id
), first_order AS (
    SELECT d.customer_id, d.order_date, d.pref_delivery_date
    FROM delivery AS d
    JOIN first_order_dates AS f
      ON d.customer_id = f.customer_id
     AND d.order_date = f.first_order_date
)
SELECT customer_id
FROM first_order
WHERE order_date = pref_delivery_date
ORDER BY customer_id;


## Query 5


In [ ]:
%%sql example-4
WITH first_order_dates AS (
    SELECT customer_id, MIN(order_date) AS first_order_date
    FROM delivery
    GROUP BY customer_id
), first_order AS (
    SELECT d.customer_id, d.order_date, d.pref_delivery_date
    FROM delivery AS d
    JOIN first_order_dates AS f
      ON d.customer_id = f.customer_id
     AND d.order_date = f.first_order_date
)
SELECT COUNT(customer_id) * 1.0 /
       NULLIF((SELECT COUNT(DISTINCT customer_id) FROM delivery), 0)
       AS immediate_percentage
FROM first_order
WHERE order_date = pref_delivery_date;


## Query 6


In [ ]:
%%sql example-5
SELECT *, ROW_NUMBER() OVER (
    PARTITION BY customer_id ORDER BY order_date
) AS order_rk
FROM delivery
ORDER BY customer_id, order_date;


## Query 7


In [ ]:
%%sql example-6
WITH ordered_delivery AS (
    SELECT *, ROW_NUMBER() OVER (
        PARTITION BY customer_id ORDER BY order_date
    ) AS order_rk
    FROM delivery
)
SELECT AVG(CASE WHEN order_date = pref_delivery_date
                THEN 1.0 ELSE 0.0 END) AS immediate_percentage
FROM ordered_delivery
WHERE order_rk = 1;


## Regression checks


In [ ]:
import math
import random

def close(actual, expected):
    assert (actual is None and expected is None) or (
        actual is not None and expected is not None and math.isclose(actual, expected, rel_tol=1e-12))

for key in ['example-0', 'example-1', 'example-2']:
    close(query_results[key][1][0][0], 2 / 3)
assert query_results['example-3'][1] == [(2,)]
for key in ['example-4', 'example-6']:
    close(query_results[key][1][0][0], 1 / 3)
expected_ranked = []
for customer in sorted({row[0] for row in delivery_fixture}):
    group = sorted(row for row in delivery_fixture if row[0] == customer)
    expected_ranked.extend((*row, rank) for rank, row in enumerate(group, 1))
assert query_results['example-5'][1] == expected_ranked

def check_fixture(users, deliveries):
    db = sqlite3.connect(':memory:')
    db.executescript(SCHEMA)
    db.executemany('INSERT INTO subscription VALUES (?, ?)', users)
    db.executemany('INSERT INTO delivery VALUES (?, ?, ?)', deliveries)
    expected_subscription = sum(premium == 'True' for _, premium in users) / len(users) if users else None
    for key in ['example-0', 'example-1', 'example-2']:
        close(db.execute(query_texts[key]).fetchone()[0], expected_subscription)
    first = {}
    for row in sorted(deliveries):
        first.setdefault(row[0], row)
    immediate = sorted(customer for customer, row in first.items() if row[1] == row[2])
    expected_delivery = len(immediate) / len(first) if first else None
    assert db.execute(query_texts['example-3']).fetchall() == [(customer,) for customer in immediate]
    for key in ['example-4', 'example-6']:
        close(db.execute(query_texts[key]).fetchone()[0], expected_delivery)
    ranked = []
    for customer in sorted(first):
        group = sorted(row for row in deliveries if row[0] == customer)
        ranked.extend((*row, rank) for rank, row in enumerate(group, 1))
    assert db.execute(query_texts['example-5']).fetchall() == ranked
    db.close()

check_fixture([], [])
check_fixture([(1, 'False')], [(1, '2023-01-01', '2023-01-02')])
check_fixture([(1, 'True')], [(1, '2023-01-01', '2023-01-01')])
check_fixture([(1, 'False'), (2, 'False')], [(1, '2023-01-01', '2023-01-02'), (2, '2023-02-01', '2023-02-02')])
check_fixture([(1, 'True'), (2, 'True')], [(1, '2023-01-01', '2023-01-01'), (2, '2023-02-01', '2023-02-01')])
check_fixture(list(reversed(subscription_fixture)), list(reversed(delivery_fixture)))
# Later immediate orders do not turn a customer's scheduled first order into an immediate one.
repeat_orders = delivery_fixture + [(1, (date(2024, 1, 1) + timedelta(days=i)).isoformat(),
                                       (date(2024, 1, 1) + timedelta(days=i)).isoformat()) for i in range(30)]
check_fixture(subscription_fixture, repeat_orders)

rng = random.Random(24)
for case in range(32):
    users = [(i, rng.choice(['True', 'False'])) for i in range(1, rng.randrange(1, 12))]
    deliveries = []
    for customer in range(1, rng.randrange(1, 10)):
        for order in range(rng.randrange(1, 6)):
            placed = date(2022, 11, 20) + timedelta(days=customer + order * 20)
            preferred = placed + timedelta(days=rng.randrange(4))
            deliveries.append((customer, placed.isoformat(), preferred.isoformat()))
    rng.shuffle(users)
    rng.shuffle(deliveries)
    check_fixture(users, deliveries)

# The source's row-based denominator is observably wrong on its own table.
wrong_denominator = 1 / len(delivery_fixture)
assert not math.isclose(wrong_denominator, query_results['example-4'][1][0][0])
def rejected(sql_text, values):
    db = sqlite3.connect(':memory:')
    db.executescript(SCHEMA)
    try:
        db.executemany(sql_text, values)
    except sqlite3.IntegrityError:
        pass
    else:
        raise AssertionError('Invalid fixture was not rejected')
    finally:
        db.close()
rejected('INSERT INTO subscription VALUES (?, ?)', [(1, None)])
rejected('INSERT INTO subscription VALUES (?, ?)', [(1, 'Maybe')])
rejected('INSERT INTO subscription VALUES (?, ?)', [(1, 'True'), (1, 'False')])
rejected('INSERT INTO delivery VALUES (?, ?, ?)', [(1, '2023-01-01', '2023-01-01'), (1, '2023-01-01', '2023-01-02')])
rejected('INSERT INTO delivery VALUES (?, ?, ?)', [(1, '01/02/2023', '02/02/2023')])
rejected('INSERT INTO delivery VALUES (?, ?, ?)', [(1, '2023-02-01', '2023-01-01')])
assert connection.execute('SELECT * FROM subscription ORDER BY user_id').fetchall() == subscription_fixture
assert connection.execute('SELECT * FROM delivery ORDER BY customer_id, order_date').fetchall() == sorted(delivery_fixture)
connection.close()
print('All seven SQL examples and full-result regressions passed.')
print('Source subscription rate: 2/3; source immediate-first-order rate: 1/3, not 1/7.')
print('Tested empty/all/none, reverse insertion order, repeated orders, 32 seeded fixtures and schema violations.')
print('Empty populations return NULL. Original source rows remained unchanged; all databases stayed in-memory.')
